# Read compiled ITCHI event

This notebook reads a compiled ITCHI event previously exported as NetCDF or Zarr. The purpose is to validate that event-level ITCHI products can be consumed without recomputing the index.

This notebook assumes that the synthetic event file was generated with:

```bash
python examples/smoke_test_synthetic.py \
  --output-path outputs/events/smoke_test_synthetic.nc
```

## 1. Imports

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt

from itchi.io import read_dataset

## 2. Input file

In [ ]:
input_path = Path('../outputs/events/smoke_test_synthetic.nc')

if not input_path.exists():
    raise FileNotFoundError(
        f'Compiled event file not found: {input_path}. '
        'Run examples/smoke_test_synthetic.py first.'
    )

input_path

## 3. Read compiled event

In [ ]:
ds = read_dataset(input_path)
ds

## 4. Inspect dimensions and variables

In [ ]:
print('Dimensions:')
for name, size in ds.sizes.items():
    print(f'  {name}: {size}')

print('\nVariables:')
for name in ds.data_vars:
    print(f'  {name}: dims={ds[name].dims}, shape={ds[name].shape}')

## 5. Inspect metadata

In [ ]:
print('Global attributes:')
for key, value in ds.attrs.items():
    if key == 'snapshot_metadata_json':
        print(f'{key}: <serialized metadata>')
    else:
        print(f'{key}: {value}')

metadata = json.loads(ds.attrs.get('snapshot_metadata_json', '[]'))
metadata

## 6. Validate required variables

In [ ]:
required_variables = ['ITCHI', 'ITCHI_max', 'ITCHI_acc']
missing = [name for name in required_variables if name not in ds]

if missing:
    raise KeyError(f'Missing required variables: {missing}')

for name in required_variables:
    min_value = float(ds[name].min(skipna=True))
    max_value = float(ds[name].max(skipna=True))
    print(f'{name}: min={min_value:.3f}, max={max_value:.3f}')

    if min_value < 0.0 or max_value > 1.0:
        raise ValueError(f'{name} is outside [0, 1].')

## 7. Validate event-product structure

In [ ]:
if 'time' not in ds['ITCHI'].dims:
    raise ValueError('ITCHI must contain a time dimension.')

if 'time' in ds['ITCHI_max'].dims:
    raise ValueError('ITCHI_max should not contain time.')

if 'time' in ds['ITCHI_acc'].dims:
    raise ValueError('ITCHI_acc should not contain time.')

print('Event-product structure is valid.')

## 8. Plot diagnostic fields

In [ ]:
fields = [
    (ds['ITCHI'].isel(time=0), 'ITCHI first snapshot'),
    (ds['ITCHI_max'], 'ITCHI_max'),
    (ds['ITCHI_acc'], 'ITCHI_acc'),
]

fig, axes = plt.subplots(1, 3, figsize=(13, 4), constrained_layout=True)

for ax, (field, title) in zip(axes, fields, strict=False):
    image = ax.pcolormesh(field.values, shading='auto')
    ax.set_title(title)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    fig.colorbar(image, ax=ax)

plt.show()

## 9. Close dataset

In [ ]:
ds.close()